# LEVI on Kaggle — free GPU, 30 h/week

**Before you run anything, in the right sidebar:**

1. **Accelerator** → `GPU P100` or `GPU T4 x2`
2. **Internet** → **On** (required to install and download weights)

Then **Run All**. With **Save & Run All (Commit)** it keeps going after you close the tab.

| | |
|---|---|
| GPU | P100 16 GB or 2 × T4 (32 GB) |
| Session | 12 hours |
| Quota | ~30 GPU h/week, resets weekly |
| RAM / cores | ~29 GB / 4 |
| Persists | `/kaggle/working` (20 GB) |
| Cost | free, no card |

An 8B model at Q4 (~5 GB) runs at roughly **25–40 tok/s** on a T4 — versus 4 tok/s on the Codespace CPU. Same corpus, ~8× faster.


In [ ]:
#@title 0 · Secrets (run this FIRST — fixes private-repo clone)
import os
try:
    from kaggle_secrets import UserSecretsClient
    _u = UserSecretsClient()
    for k in ("GITHUB_TOKEN", "LEVI_SYNC_REPO", "GROQ_API_KEY"):
        try:
            v = _u.get_secret(k)
            if v: os.environ[k] = v
        except Exception:
            pass
except Exception:
    pass

for k in ("GITHUB_TOKEN", "LEVI_SYNC_REPO", "GROQ_API_KEY"):
    v = os.environ.get(k, "")
    print(f"{k:16} {'set (' + v[:4] + '...' + v[-4:] + ')' if v else 'MISSING'}")
print()
print("GITHUB_TOKEN needs: repo scope on AmbitiousBoyII14/levi + levi-runs")
print("Add them: File -> 'Add-ons' -> Secrets (left sidebar)")


In [ ]:
#@title 1 · Detect hardware
import subprocess, os, sys
def sh(c, **kw):
    return subprocess.run(c, shell=True, capture_output=True, text=True, **kw).stdout.strip()

print("python ", sys.version.split()[0])
print("cores  ", os.cpu_count())
print("ram    ", sh("free -g | awk '/Mem:/{print $2}'"), "GB")
print("disk   ", sh("df -h /kaggle/working | tail -1 | awk '{print $4}'"), "free")
print()
gpu = sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader")
print(gpu or "NO GPU -- check Accelerator in the right sidebar")
print()
print("internet:", sh("curl -s -o /dev/null -w '%{http_code}' --max-time 10 https://github.com"))


In [ ]:
#@title 2 · Install ollama
import subprocess, os, time
if not os.path.exists("/usr/local/bin/ollama"):
    print(subprocess.run("curl -fsSL https://ollama.com/install.sh | sh",
                         shell=True, capture_output=True, text=True).stdout[-1500:])
else:
    print("ollama already present")

# serve in the background
os.environ["OLLAMA_MODELS"] = "/kaggle/working/.ollama"     # survives the session
os.environ["OLLAMA_KEEP_ALIVE"] = "30m"
subprocess.Popen("nohup ollama serve > /kaggle/working/ollama.log 2>&1 &", shell=True)
for i in range(40):
    time.sleep(1)
    if subprocess.run("curl -s --max-time 2 http://127.0.0.1:11434/api/tags",
                      shell=True, capture_output=True).returncode == 0:
        print("server up after", i+1, "s"); break
else:
    print("server did not start — tail /kaggle/working/ollama.log")


In [ ]:
#@title 3 · Pull a model (fits a T4/P100 16 GB)
MODEL = "qwen3:8b"   #@param ["qwen3:8b", "qwen3:4b", "qwen3:14b", "hf.co/llmfan46/gemma-4-E4B-it-ultra-uncensored-heretic-GGUF:Q4_K_M"]
import subprocess, time
t0 = time.time()
r = subprocess.run(f"ollama pull {MODEL}", shell=True, capture_output=True, text=True)
print(f"{MODEL} ready in {time.time()-t0:.0f}s" if r.returncode == 0 else r.stderr[-800:])

# prove the GPU is being used, not the CPU
out = subprocess.run(
    f'''curl -s http://127.0.0.1:11434/v1/chat/completions -H 'Content-Type: application/json' '''
    f'''-d '{{"model":"{MODEL}","messages":[{{"role":"user","content":"Reply with exactly: READY"}}],"max_tokens":16}}' ''',
    shell=True, capture_output=True, text=True).stdout
print(out[:400])


In [ ]:
#@title 4 · Speed test — how long will the real job take?
import subprocess, time, json
prompt = "List five common patterns you look for when reversing a C++ game engine."
t0 = time.time()
r = subprocess.run(
    f'''curl -s http://127.0.0.1:11434/v1/chat/completions -H 'Content-Type: application/json' '''
    f'''-d '{json.dumps({"model":MODEL,"messages":[{"role":"user","content":prompt}],"max_tokens":300})}' ''',
    shell=True, capture_output=True, text=True).stdout
el = time.time() - t0
try:
    d = json.loads(r); toks = d["usage"]["completion_tokens"]
    print(f"{toks} tokens in {el:.1f}s  ->  {toks/el:.1f} tok/s")
except Exception:
    print(r[:400])
print()
print(subprocess.run("nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout or "no nvidia-smi")


In [ ]:
#@title 5 · Get the corpus — upload a Kaggle Dataset first
import os, subprocess
# Easiest path: on kaggle.com click "Datasets" -> New Dataset -> drag your zip in
# -> publish -> then "Add Data" on this notebook and it appears under
#     /kaggle/input/<your-dataset-name>/
os.makedirs("/kaggle/working/work", exist_ok=True)
print(subprocess.run("ls -1 /kaggle/input 2>/dev/null || echo '(nothing attached)'", shell=True, capture_output=True, text=True).stdout)
print(subprocess.run("find /kaggle/input -name '*.zip' -o -name '*.tar*' 2>/dev/null | head", shell=True, capture_output=True, text=True).stdout or "(no archives found)")


In [ ]:
#@title 6 · Clone LEVI + index (free, no model calls)
import subprocess, os
REPO = "AmbitiousBoyII14/levi"
TOK  = os.environ.get("GITHUB_TOKEN", "")
url  = f"https://{TOK}@github.com/{REPO}.git" if TOK else f"https://github.com/{REPO}.git"

if not os.path.exists("/kaggle/working/levi"):
    r = subprocess.run(f"git clone --depth 1 {url} /kaggle/working/levi",
                       shell=True, capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip()[:600])
    if "could not read Username" in (r.stderr or "") or r.returncode != 0:
        print()
        print("PRIVATE REPO — needs GITHUB_TOKEN.")
        print("  File -> Add-ons -> Secrets -> add GITHUB_TOKEN (classic token, 'repo' scope)")
        print("  then re-run cell 0 and this cell.")
else:
    print(subprocess.run("cd /kaggle/working/levi && git pull",
                         shell=True, capture_output=True, text=True).stdout.strip() or "up to date")

ZIP = subprocess.run("find /kaggle/input -name '*.zip' 2>/dev/null | head -1",
                     shell=True, capture_output=True, text=True).stdout.strip()
if os.path.isdir("/kaggle/working/levi"):
    if ZIP:
        print(subprocess.run(f"cd /kaggle/working/levi && LEVI_WORK=/kaggle/working/work "
                             f"LEVI_OUT=/kaggle/working/index bash kit/levi-index.sh '{ZIP}'",
                             shell=True, capture_output=True, text=True).stdout[-1200:])
    else:
        print("no zip found — attach a Dataset first (cell 5)")


## If the session died — resume in the next one

`/kaggle/working` is wiped when an interactive session dies. What saves you:

1. **Auto-sync** (cell 7) pushes evidence to GitHub every 3 minutes.
2. **`state.json`** records finished batches. `--resume` skips them.
3. Model weights stay in `/kaggle/working/.ollama` *within a session*; across
   sessions they re-download (~5 GB, a few minutes).

New session, run in order:

```
cell 1  detect hardware
cell 2  install ollama
cell 3  pull model            (re-downloads — unavoidable)
cell 7b restore from GitHub   <-- the cell below
cell 6  clone + index
cell 7  run with --resume
```

## The 12-hour rules, plainly

| What you do | What survives |
|---|---|
| Interactive session, let it die | **nothing** unless auto-sync ran |
| Interactive, clicked **Save Version** | whatever existed at that click |
| **Save & Run All (Commit)** hits 12 h | **everything** — outputs saved automatically |
| Commit run crashes early | whatever it wrote before the crash |

**Commit is the safe mode.** For anything longer than a couple of hours, use
*Save & Run All (Commit)* instead of running interactively: it starts from a
clean state, runs headless after you close the tab, and saves `/kaggle/working`
as the version's output at the end — including on a timeout.


In [ ]:
#@title 7b · Restore evidence from GitHub (run FIRST in a new session)
import subprocess
r = subprocess.run("python3 /kaggle/working/levi/kaggle/sync.py pull",
                   shell=True, capture_output=True, text=True)
print(r.stdout or r.stderr)
r = subprocess.run("python3 /kaggle/working/levi/kaggle/sync.py status",
                   shell=True, capture_output=True, text=True)
print(r.stdout or r.stderr)


In [ ]:
#@title 7 · Run the agent — checkpoints + auto-sync
import subprocess, os, time, threading, pathlib
CORPUS = "/kaggle/working/work/extract"
OUT    = "/kaggle/working/evidence"
SYNC_EVERY = 180       # seconds between GitHub pushes
os.makedirs(OUT, exist_ok=True)

if not os.path.isdir(CORPUS):
    print("no corpus at", CORPUS, "— run cells 5 and 6")
else:
    cmd = (f"cd /kaggle/working/levi && python3 kit/agent.py "
           f"--prompt prompts/levi-v2-operator.md "
           f"--work {CORPUS} --out {OUT} "
           f"--provider ollama --model {MODEL} "
           f"--base http://127.0.0.1:11434/v1 "
           f"--max-chars 9000 --pace 0 --budget-calls 60 --resume")
    print(cmd, "\n")

    def sync_loop(stop):
        """Keep pushing evidence to GitHub while the run proceeds."""
        while not stop.is_set():
            stop.wait(SYNC_EVERY)
            if stop.is_set(): break
            r = subprocess.run("python3 /kaggle/working/levi/kaggle/sync.py push",
                               shell=True, capture_output=True, text=True)
            print("[sync]", (r.stdout or r.stderr).strip()[:160], flush=True)

    stop = threading.Event()
    t = threading.Thread(target=sync_loop, args=(stop,), daemon=True)
    has_token = bool(os.environ.get("GITHUB_TOKEN") or os.environ.get("LEVI_SYNC_REPO"))
    if has_token:
        t.start(); print(f"auto-sync every {SYNC_EVERY}s\n")
    else:
        print("no GITHUB_TOKEN / LEVI_SYNC_REPO -> running WITHOUT backup")
        print("add them as Kaggle Secrets to survive a dead session\n")

    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    stop.set()

    print("\nexit", p.returncode)
    if has_token:
        r = subprocess.run("python3 /kaggle/working/levi/kaggle/sync.py push",
                           shell=True, capture_output=True, text=True)
        print("[final sync]", (r.stdout or r.stderr).strip())


## 8 · Read the result

`/kaggle/working/evidence/` holds the output, and `/kaggle/working` is saved as the
notebook's output — so it survives after the session ends.

- `report.md` — the analysis
- `ledger.txt` — sha256 + byte counts, proof of what was actually read
- `batch_NN.json` — per-batch record
- `state.json` — feeds `--resume`

```python
print(open("/kaggle/working/evidence/report.md").read()[:4000])
```

## Keeping it going past 12 hours

Kaggle kills the session at 12 h. The job is resumable:

1. It writes `state.json` as it goes
2. Next session, re-run cells 2–7 — `--resume` skips completed batches
3. Weights stay in `/kaggle/working/.ollama`, so no re-download

With **Save & Run All (Commit)** the notebook runs headless after you close the tab.

## Weekly budget

~30 GPU h/week ≈ 2.5 full 12-hour sessions. Enough for a serious sweep if you
batch selectively instead of reading everything.
